In [2]:
# needed imports 
import geopandas as gpd
import pandas as pd
from pathlib import Path
import numpy as np


# get the current dir
current_dir = Path.cwd()

# print the current dir
print(f"Current dir is: {current_dir}")

Current dir is: c:\dev\projects\respiratory_project\code\notebooks


In [4]:
# get the data dir
data_dir = Path(f"{current_dir.parent.parent}/data")

# print the data dir
print("Data dir:", data_dir)

Data dir: c:\dev\projects\respiratory_project\data


In [4]:
# get the tracts data
tracts_gdf = gpd.read_file(f"{data_dir}/raw/tracts/tl_2025_53_tract.shp")

# print the data columns 
print(tracts_gdf.columns)

Index(['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'GEOIDFQ', 'NAME',
       'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT',
       'INTPTLON', 'geometry'],
      dtype='str')


In [5]:
tracts_to_keep_columns = [
    "GEOID",
    "COUNTYFP",
    "STATEFP",
    "NAME",
    "ALAND",
    "AWATER",
    "geometry"
]

tracts_gdf = tracts_gdf[tracts_to_keep_columns]

tracts_gdf.head()

,GEOID,COUNTYFP,STATEFP,NAME,ALAND,AWATER,geometry
0,53071920301,071,53,9203.01,2621059,0,"POLYGON ((-118.38863 46.03534, -118.38861 46.0..."
1,53071920302,071,53,9203.02,2919497,0,"POLYGON ((-118.39905 46.04588, -118.39905 46.0..."
2,53071920901,071,53,9209.01,21809260,189083,"POLYGON ((-118.31474 46.05855, -118.31474 46.0..."
3,53033030003,033,53,300.03,4604664,963078,"POLYGON ((-122.33104 47.37207, -122.32465 47.3..."
4,53033005307,033,53,53.07,130030,0,"POLYGON ((-122.31083 47.66307, -122.3108 47.66..."


In [6]:
# keep only the study are 

tracts_gdf = tracts_gdf[(tracts_gdf["COUNTYFP"] == "033") & (tracts_gdf["STATEFP"] == "53")]

tracts_gdf.shape

(495, 7)

In [7]:
# santy check 

print(f"total null:", tracts_gdf.isna().sum(), sep="\n")
print("*" * 30)
print(f"total duplicates:", tracts_gdf.duplicated().sum())

total null:
GEOID       0
COUNTYFP    0
STATEFP     0
NAME        0
ALAND       0
AWATER      0
geometry    0
dtype: int64
******************************
total duplicates: 0


In [8]:
# save the study area dataset

tracts_gdf.to_file(str(data_dir) + "/processed/" + "kc_tracts.geojson", driver='GeoJSON')

In [9]:
# # get the places dataset & clean it

places_keep = [
    "StateAbbr",
    "CountyName",
    "CountyFIPS",
    "TractFIPS",
    "TotalPopulation",
    "TotalPop18plus",
    "CASTHMA_CrudePrev",
    "CASTHMA_Crude95CI"
]

places = pd.read_csv(f"{data_dir}/raw/places_asthma/PLACES__Census_Tract_Data__GIS_Friendly_Format___2024_release.csv", usecols=places_keep)

# Force FIPS into correctly padded strings
places["CountyFIPS"] = places["CountyFIPS"].astype("Int64").astype(str).str.zfill(5)
places["TractFIPS"]  = places["TractFIPS"].astype("Int64").astype(str).str.zfill(11)

# Filter to King County, WA
kc_places = places[(places["StateAbbr"] == "WA") & (places["CountyFIPS"] == "53033")].copy()

# Rename tract key to match shapefile GEOID
kc_places = kc_places.rename(columns={"TractFIPS": "GEOID"})

# Basic QA
print("KC PLACES rows:", len(kc_places))
print("GEOID length (unique):", kc_places["GEOID"].str.len().unique())
print("Example GEOIDs:", kc_places["GEOID"].head(3).tolist())
print("Asthma min/max:", kc_places["CASTHMA_CrudePrev"].min(), kc_places["CASTHMA_CrudePrev"].max())
print("Missing asthma:", kc_places["CASTHMA_CrudePrev"].isna().sum())


kc_places.to_csv(str(data_dir) + "/processed/asthma_data.csv", index=False)


KC PLACES rows: 494
GEOID length (unique): [11]
Example GEOIDs: ['53033000101', '53033000102', '53033000201']
Asthma min/max: 7.3 13.9
Missing asthma: 0


In [57]:
# load places + tractes to join dataset

places_df = pd.read_csv(f"{data_dir}/processed/asthma_data.csv")
tracts_gdf = gpd.read_file(f"{data_dir}/processed/kc_tracts.geojson")

# ensure both join columns in the same type

places_df["GEOID"] = places_df["GEOID"].astype(str)
tracts_gdf["GEOID"] = tracts_gdf["GEOID"].astype(str)

# print both datasets shapes

places_df.shape , tracts_gdf.shape

((494, 8), (495, 7))

In [61]:
# preform the join on GEOID

joind_data = pd.merge(tracts_gdf, places_df, on="GEOID", how="left")

joind_data.head()

,GEOID,COUNTYFP,STATEFP,NAME,ALAND,AWATER,geometry,StateAbbr,CountyName,CountyFIPS,TotalPopulation,TotalPop18plus,CASTHMA_CrudePrev,CASTHMA_Crude95CI
0,53033030003,033,53,300.03,4604664,963078,"POLYGON ((-122.33104 47.37207, -122.32465 47.3...",WA,King,53033.0,6592.0,5347.0,11.1,"(10.3, 12.0)"
1,53033005307,033,53,53.07,130030,0,"POLYGON ((-122.31083 47.66307, -122.3108 47.66...",WA,King,53033.0,2921.0,2824.0,13.9,"(12.7, 15.3)"
2,53033030309,033,53,303.09,2194432,27034,"POLYGON ((-122.35084 47.29868, -122.34963 47.2...",WA,King,53033.0,5906.0,4666.0,10.6,"( 9.8, 11.5)"
3,53033030310,033,53,303.10,3205495,0,"POLYGON ((-122.36115 47.2864, -122.36114 47.28...",WA,King,53033.0,6466.0,4998.0,9.6,"( 8.8, 10.4)"
4,53033030311,033,53,303.11,1900456,3993,"POLYGON ((-122.37697 47.30796, -122.37695 47.3...",WA,King,53033.0,5093.0,3810.0,10.4,"( 9.6, 11.2)"


In [ ]:
# keep the needed columns only
joind_data = joind_data[["GEOID", "CASTHMA_CrudePrev", "CASTHMA_Crude95CI", "TotalPopulation", "TotalPop18plus" , "geometry"]]

In [63]:
joind_data

,GEOID,CASTHMA_CrudePrev,CASTHMA_Crude95CI,TotalPopulation,TotalPop18plus,geometry
0,53033030003,11.1,"(10.3, 12.0)",6592.0,5347.0,"POLYGON ((-122.33104 47.37207, -122.32465 47.3..."
1,53033005307,13.9,"(12.7, 15.3)",2921.0,2824.0,"POLYGON ((-122.31083 47.66307, -122.3108 47.66..."
2,53033030309,10.6,"( 9.8, 11.5)",5906.0,4666.0,"POLYGON ((-122.35084 47.29868, -122.34963 47.2..."
3,53033030310,9.6,"( 8.8, 10.4)",6466.0,4998.0,"POLYGON ((-122.36115 47.2864, -122.36114 47.28..."
4,53033030311,10.4,"( 9.6, 11.2)",5093.0,3810.0,"POLYGON ((-122.37697 47.30796, -122.37695 47.3..."
...,...,...,...,...,...,...
490,53033025304,9.7,"( 8.9, 10.5)",3733.0,3106.0,"POLYGON ((-122.21913 47.52372, -122.21876 47.5..."
491,53033001201,10.6,"( 9.8, 11.4)",3781.0,3279.0,"POLYGON ((-122.33406 47.71229, -122.33332 47.7..."
492,53033001400,10.1,"( 9.4, 11.0)",5298.0,4360.0,"POLYGON ((-122.38442 47.71136, -122.38367 47.7..."
493,53033001600,9.5,"( 8.8, 10.3)",4504.0,3534.0,"POLYGON ((-122.39218 47.70656, -122.3893 47.70..."


In [ ]:
# rename columns for better understanding
columns_names = {
    "CASTHMA_CrudePrev" : "asthma_prev",
    "CASTHMA_Crude95CI" : "asthmai_ci",
    "TotalPopulation" : "pop_total", 
    "TotalPop18plus" : "pop_18plus"
}

joind_data.rename(columns= columns_names, inplace=True)

In [ ]:
# check null counts
joind_data.isna().sum()

GEOID          0
asthma_prev    1
asthmai_ci     1
pop_total      1
pop_18plus     1
geometry       0
dtype: int64

In [73]:
# Identify the missing tract
missing_row = joind_data[joind_data["asthma_prev"].isna()]
missing_row

,GEOID,asthma_prev,asthmai_ci,pop_total,pop_18plus,geometry
195,53033990100,NaN,NaN,NaN,NaN,"POLYGON ((-122.54166 47.34919, -122.54139 47.3..."


In [ ]:
# drop the null row
joind_data.dropna(inplace=True)

In [ ]:
# adding columns and fixing data type
joind_data["pop_total"] = joind_data["pop_total"].astype(int)
joind_data["pop_18plus"] = joind_data["pop_18plus"].astype(int)
joind_data["GEOID"] = joind_data["GEOID"].astype(str)
joind_data["asthmai_ci"] = joind_data["asthmai_ci"].str.replace("[($)]", "" , regex=True)
joind_data["asthma_ci_low"] = joind_data["asthmai_ci"].apply(lambda r : r.split(",")[0]).astype(float)
joind_data["asthma_ci_high"] = joind_data["asthmai_ci"].apply(lambda r : r.split(",")[1]).astype(float)

In [ ]:
# remove unwanted column
joind_data.drop(columns= ["asthmai_ci"], inplace= True)

In [112]:
joind_data.isna().sum()

GEOID             0
asthma_prev       0
pop_total         0
pop_18plus        0
geometry          0
asthma_ci_low     0
asthma_ci_high    0
dtype: int64

#### Save checkpoint: Curated base table (NOT the final dataset)

In [114]:
joind_data.to_file(str(data_dir) + "/processed/" + "kc_master_phaseA.geojson", driver='GeoJSON')

In [162]:
# load master phaseA
master = gpd.read_file(f"{data_dir}/processed/kc_master_phaseA.geojson")

# load roads dataset 
roads = gpd.read_file(f"{data_dir}/raw/osm_geofabrik/gis_osm_roads_free_1.shp")

# load landuse dataset 
landuse = gpd.read_file(f"{data_dir}/raw/osm_geofabrik/gis_osm_landuse_a_free_1.shp")

# load natural_a dataset 
natural_a = gpd.read_file(f"{data_dir}/raw/osm_geofabrik/gis_osm_natural_a_free_1.shp")

In [163]:
# santy check
print("roads cols:", roads.columns.tolist())
print("landuse cols:", landuse.columns.tolist())
print("natural_a cols:", natural_a.columns.tolist())
print("roads CRS:", roads.crs)

roads cols: ['osm_id', 'code', 'fclass', 'name', 'ref', 'oneway', 'maxspeed', 'layer', 'bridge', 'tunnel', 'geometry']
landuse cols: ['osm_id', 'code', 'fclass', 'name', 'geometry']
natural_a cols: ['osm_id', 'code', 'fclass', 'name', 'geometry']
roads CRS: EPSG:4326


In [164]:
# geom type check
print("roads geom types:", roads.geom_type.value_counts().head(), sep = "\n")
print("landuse geom types:", landuse.geom_type.value_counts().head(), sep = "\n")
print("natural_a geom types:", natural_a.geom_type.value_counts().head(), sep = "\n")

roads geom types:
LineString    1413654
Name: count, dtype: int64
landuse geom types:
Polygon         161594
MultiPolygon      2126
Name: count, dtype: int64
natural_a geom types:
Polygon         2114
MultiPolygon     136
Name: count, dtype: int64


In [165]:
# project master tracts to meters (UTM 10N) 
TARGET_CRS = "EPSG:26910"
kc = master.to_crs(TARGET_CRS)

# bounding box of King County tracts (in projected meters)
minx, miny, maxx, maxy = kc.total_bounds
bbox = (minx, miny, maxx, maxy)

print("Target CRS:", TARGET_CRS)
print("KC bbox (meters):", bbox)
print("KC tracts:", len(kc))

# Read OSM layers and reproject 
roads   = roads[["osm_id", "fclass", "geometry"]].to_crs(TARGET_CRS)
landuse = landuse[["osm_id", "fclass", "geometry"]].to_crs(TARGET_CRS)
natural_a = natural_a[["osm_id", "fclass", "geometry"]].to_crs(TARGET_CRS)

# Fast bbox filter (dramatically reduces size) 
roads_kc   = roads.cx[minx:maxx, miny:maxy].copy()
landuse_kc = landuse.cx[minx:maxx, miny:maxy].copy()
natural_kc = natural_a.cx[minx:maxx, miny:maxy].copy()

print("\nOSM counts after bbox filter:")
print("roads_kc:", len(roads_kc))
print("landuse_kc:", len(landuse_kc))
print("natural_kc:", len(natural_kc))

# Inspect available classes (so we define features correctly) 
print("\nTop road fclass values:")
print(roads_kc["fclass"].value_counts().head(25))

print("\nTop landuse fclass values:")
print(landuse_kc["fclass"].value_counts().head(25))

print("\nTop natural fclass values:")
print(natural_kc["fclass"].value_counts().head(25))


Target CRS: EPSG:26910
KC bbox (meters): (np.float64(535349.0909352319), np.float64(5215717.029434288), np.float64(645077.9297652419), np.float64(5293554.045167913))
KC tracts: 494

OSM counts after bbox filter:
roads_kc: 533003
landuse_kc: 41588
natural_kc: 272

Top road fclass values:
fclass
footway           222206
service           175848
residential        64755
secondary          11913
path               10956
tertiary            9876
steps               6305
track               6279
cycleway            6064
primary             6032
motorway_link       3017
unclassified        2585
motorway            2336
pedestrian          1050
trunk                872
secondary_link       479
primary_link         439
living_street        425
bridleway            388
tertiary_link        258
track_grade5         195
track_grade3         188
track_grade2         187
track_grade4         152
trunk_link           112
Name: count, dtype: int64

Top landuse fclass values:
fclass
residential        

In [166]:
# select driveable roads
ROAD_FCLASS_DRIVE = {
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link",
    "residential", "unclassified", "living_street",
    "pedestrian"
}

# select green spaces
GREEN_FCLASS = {
    "forest", "grass", "park", "meadow", "scrub",
    "recreation_ground", "nature_reserve", "heath",
    "orchard", "allotments", "cemetery"
}

# filter to the selected sets
roads_drive = roads_kc[roads_kc["fclass"].isin(ROAD_FCLASS_DRIVE)]
green_poly = landuse_kc[landuse_kc["fclass"].isin(GREEN_FCLASS)]

print("roads_kc total:", len(roads_kc))
print("roads_drive kept:", len(roads_drive))
print("landuse_kc total:", len(landuse_kc))
print("green_poly kept:", len(green_poly))

print("\nTop roads_drive fclass:")
print(roads_drive["fclass"].value_counts().head(15))

print("\nTop green fclass:")
print(green_poly["fclass"].value_counts().head(15))

roads_kc total: 533003
roads_drive kept: 104149
landuse_kc total: 41588
green_poly kept: 25093

Top roads_drive fclass:
fclass
residential       64755
secondary         11913
tertiary           9876
primary            6032
motorway_link      3017
unclassified       2585
motorway           2336
pedestrian         1050
trunk               872
secondary_link      479
primary_link        439
living_street       425
tertiary_link       258
trunk_link          112
Name: count, dtype: int64

Top green fclass:
fclass
grass                9900
forest               8294
park                 3139
scrub                2109
meadow                650
recreation_ground     365
allotments            247
nature_reserve        235
cemetery               74
heath                  52
orchard                28
Name: count, dtype: int64


In [85]:
roads_drive.head()

,osm_id,fclass,geometry
0,4634293,motorway,"LINESTRING (551241.039 5276818.536, 551247.04 ..."
1,4634309,motorway_link,"LINESTRING (550880.404 5276348.635, 550889.204..."
2,4634847,motorway_link,"LINESTRING (552204.186 5276900.858, 552214.16 ..."
3,4635028,motorway_link,"LINESTRING (551906.287 5276888.126, 551971.272..."
4,4636105,motorway_link,"LINESTRING (552207.919 5277014.795, 552186.01 ..."


In [86]:
green_poly.head()

,osm_id,fclass,geometry
0,4681382,recreation_ground,"POLYGON ((548676.997 5229822.683, 548799.017 5..."
1,4755066,park,"POLYGON ((548441.652 5274248.283, 548441.734 5..."
2,4835090,grass,"POLYGON ((549923.536 5277112.933, 549980.42 52..."
3,4848414,forest,"POLYGON ((549406.296 5280886.028, 549407.696 5..."
7,5098080,recreation_ground,"POLYGON ((550383.14 5281141.478, 550428.829 52..."


In [167]:
# Keep only needed columns (speed)
tracts = kc[["GEOID", "geometry"]].copy()
roads  = roads_drive[["fclass", "geometry"]].copy()

# Spatial join (FAST): match each road to tracts it intersects
road_pairs = gpd.sjoin(
    roads,
    tracts[["GEOID", "geometry"]],
    how="inner",
    predicate="intersects"
).reset_index(drop=True)

# Attach tract geometry so we can clip precisely
road_pairs = road_pairs.merge(
    tracts[["GEOID", "geometry"]].rename(columns={"geometry": "tract_geom"}),
    on="GEOID",
    how="left"
).drop(columns = ["index_right"])


# Clip roads to tract boundary (ACCURATE): intersection geometry
# clip_geom = road ∩ tract
road_pairs["clip_geom"] = road_pairs.geometry.intersection(road_pairs["tract_geom"])
road_pairs = road_pairs[~road_pairs["clip_geom"].is_empty].copy()

road_pairs["roads_length_m_drive"] = road_pairs.clip_geom.length 
road_len_by_tract_m = road_pairs.groupby("GEOID")["roads_length_m_drive"].sum()

In [168]:
# sanity check 
print("road_len_by_tract_m length:", len(road_len_by_tract_m))
print("unique GEOID in result:", road_len_by_tract_m.index.nunique())
print("min/max meters:", road_len_by_tract_m.min(), road_len_by_tract_m.max())

road_len_by_tract_m length: 494
unique GEOID in result: 494
min/max meters: 2004.6228705998622 471284.22434270504


In [169]:
# Convert Series -> DataFrame with a clear column name
roads_by_tract = road_len_by_tract_m.rename("roads_length_m_drive").reset_index()

# Merge (keep all 494 tracts)
kc = kc.merge(roads_by_tract, on="GEOID", how="left")
kc["roads_length_m_drive"] = kc["roads_length_m_drive"].fillna(0.0)

# Quick check
print("rows:", len(kc), "| unique GEOID:", kc["GEOID"].nunique())
print("missing roads_length_m_drive:", kc["roads_length_m_drive"].isna().sum())
kc[["GEOID", "roads_length_m_drive"]].head()


rows: 494 | unique GEOID: 494
missing roads_length_m_drive: 0


,GEOID,roads_length_m_drive
0,53033030003,42584.740622
1,53033005307,2090.239238
2,53033030309,22296.721590
3,53033030310,32094.396080
4,53033030311,23152.827072


In [135]:
# convert meters -> km
kc["roads_km_drive"] = kc["roads_length_m_drive"] / 1_000.0

# tract area (km²) — only compute if you don't already have it
if "tract_area_km2" not in kc.columns:
    kc["tract_area_km2"] = kc.geometry.area / 1_000_000.0  # m² -> km²

# density: km of road per km² of land area
kc["roads_km_per_km2_drive"] = kc["roads_km_drive"] / kc["tract_area_km2"]

# quick QA
print("roads_km_per_km2_drive min/max:",
      kc["roads_km_per_km2_drive"].min(),
      kc["roads_km_per_km2_drive"].max())
kc[["GEOID", "roads_km_drive", "tract_area_km2", "roads_km_per_km2_drive"]].head()

roads_km_per_km2_drive min/max: 0.33250649457735176 31.87131387007276


,GEOID,roads_km_drive,tract_area_km2,roads_km_per_km2_drive
0,53033030003,42.584741,5.563658,7.654090
1,53033005307,2.090239,0.129932,16.087138
2,53033030309,22.296722,2.219825,10.044361
3,53033030310,32.094396,3.203117,10.019740
4,53033030311,23.152827,1.903031,12.166291


In [170]:
# spatial join to add geoid
green_pairs = gpd.sjoin(
    green_poly[["osm_id","fclass","geometry"]],
    tracts[["GEOID","geometry"]],
    how="inner",
    predicate="intersects"
).reset_index(drop=True).drop(columns=["index_right"])


green_pairs.head()

,osm_id,fclass,geometry,GEOID
0,4755066,park,"POLYGON ((548441.652 5274248.283, 548441.734 5...",53033007101
1,4755066,park,"POLYGON ((548441.652 5274248.283, 548441.734 5...",53033007203
2,4835090,grass,"POLYGON ((549923.536 5277112.933, 549980.42 52...",53033005401
3,4848414,forest,"POLYGON ((549406.296 5280886.028, 549407.696 5...",53033004600
4,5098080,recreation_ground,"POLYGON ((550383.14 5281141.478, 550428.829 52...",53033004600


In [171]:
green_pairs = green_pairs.merge(
    tracts[["GEOID", "geometry"]].rename(columns={"geometry": "tract_geom"}),
    on="GEOID",
    how="left"
)

green_pairs[["osm_id", "fclass", "GEOID"]].head()

,osm_id,fclass,GEOID
0,4755066,park,53033007101
1,4755066,park,53033007203
2,4835090,grass,53033005401
3,4848414,forest,53033004600
4,5098080,recreation_ground,53033004600


In [172]:
# sanity check
print("rows green_pairs:", len(green_pairs))
print("missing tract_geom:", green_pairs["tract_geom"].isna().sum())
print("tract_geom types:", green_pairs["tract_geom"].geom_type.value_counts().head())

rows green_pairs: 18749
missing tract_geom: 0
tract_geom types: Polygon    18749
Name: count, dtype: int64


In [173]:
# Clip greenspace polygon to tract boundary (accurate)
green_pairs["clip_geom"] = green_pairs.geometry.intersection(green_pairs["tract_geom"])
green_pairs = green_pairs[~green_pairs["clip_geom"].is_empty].copy()

# Keep only polygon outputs (robust)
green_pairs = green_pairs[green_pairs["clip_geom"].geom_type.isin(["Polygon", "MultiPolygon"])].copy()

# Area inside tract (m^2 because CRS is EPSG:26910 meters)
green_pairs["green_area_m2"] = green_pairs["clip_geom"].area

green_pairs[["GEOID", "fclass", "green_area_m2"]].head()

,GEOID,fclass,green_area_m2
0,53033007101,park,289848.158506
1,53033007203,park,14235.682617
2,53033005401,grass,3740.699471
3,53033004600,forest,2159.925940
4,53033004600,recreation_ground,1685.773145


In [174]:
green_area_by_tract = (
    green_pairs.groupby("GEOID")["green_area_m2"]
    .sum()
    .rename("green_area_m2")
    .reset_index()
)

print("Tracts with some greenspace:", len(green_area_by_tract))
green_area_by_tract.head()


Tracts with some greenspace: 492


,GEOID,green_area_m2
0,53033000101,5459.542471
1,53033000102,139268.833026
2,53033000201,431464.794930
3,53033000202,7333.252779
4,53033000300,44051.526940


In [175]:
kc = kc.merge(green_area_by_tract, on="GEOID", how="left")
kc["green_area_m2"] = kc["green_area_m2"].fillna(0.0)

print("kc rows:", len(kc), "| unique GEOID:", kc["GEOID"].nunique())
print("missing green_area_m2:", kc["green_area_m2"].isna().sum())
kc[["GEOID", "green_area_m2"]].head()

kc rows: 494 | unique GEOID: 494
missing green_area_m2: 0


,GEOID,green_area_m2
0,53033030003,1.315955e+06
1,53033005307,2.882592e+03
2,53033030309,5.409063e+05
3,53033030310,4.321418e+05
4,53033030311,1.861624e+05


In [176]:
kc["tract_area_m2"] = kc.geometry.area

# greenspace share of tract area (0 to 1 in an ideal non-overlapping world)
kc["green_pct"] = np.where(
    kc["tract_area_m2"] > 0,
    kc["green_area_m2"] / kc["tract_area_m2"],
    np.nan
)

print("green_pct min/max:", kc["green_pct"].min(), kc["green_pct"].max())
print("tracts with green_pct > 1:", (kc["green_pct"] > 1).sum())

kc.loc[kc["green_pct"] > 1, ["GEOID", "green_area_m2", "tract_area_m2", "green_pct"]] \
  .sort_values("green_pct", ascending=False) \
  .head(10)

green_pct min/max: 0.0 1.43217542196084
tracts with green_pct > 1: 4


,GEOID,green_area_m2,tract_area_m2,green_pct
387,53033032102,1.142127e+08,7.974767e+07,1.432175
303,53033025006,2.939485e+07,2.239439e+07,1.312599
401,53033032104,2.647545e+07,2.226463e+07,1.189126
341,53033032215,1.823370e+07,1.595152e+07,1.143070


In [155]:
# --- Fix double-counting: union greenspace within each tract, then compute area ---

# Use the clipped geometry as the active geometry
green_clip = green_pairs[["GEOID", "clip_geom"]].copy()
green_clip = green_clip.set_geometry("clip_geom")

# (Optional but recommended) clean invalid geometries to avoid weird area results
# If your geopandas has make_valid:
try:
    green_clip["clip_geom"] = green_clip["clip_geom"].make_valid()
except Exception:
    # fallback: buffer(0) trick
    green_clip["clip_geom"] = green_clip["clip_geom"].buffer(0)

# Dissolve = union all green pieces inside each tract (removes overlaps)
green_union = green_clip.dissolve(by="GEOID")

# Now area is not double-counted
green_area_by_tract = (
    green_union.geometry.area
    .rename("green_area_m2")
    .reset_index()
)

print("Tracts with unioned greenspace:", len(green_area_by_tract))
green_area_by_tract.head()

Tracts with unioned greenspace: 492


,GEOID,green_area_m2
0,53033000101,5459.542471
1,53033000102,139199.406907
2,53033000201,400320.909581
3,53033000202,7333.252779
4,53033000300,44051.526940


In [179]:
kc = kc.drop(columns=["green_area_m2"]).merge(green_area_by_tract, on="GEOID", how="left")
kc["green_area_m2"] = kc["green_area_m2"].fillna(0.0)

kc["tract_area_m2"] = kc.geometry.area
kc["green_pct"] = np.where(
    kc["tract_area_m2"] > 0,
    kc["green_area_m2"] / kc["tract_area_m2"],
    np.nan
)

print("green_pct min/max:", kc["green_pct"].min(), kc["green_pct"].max())
print("tracts with green_pct > 1:", (kc["green_pct"] > 1).sum())

green_pct min/max: 0.0 1.43217542196084
tracts with green_pct > 1: 4


In [181]:
# FIX: avoid double-counting overlaps by unioning within each tract 

# 1) union all clipped greenspace geometries per tract
green_union_by_tract = (
    green_pairs.groupby("GEOID")["clip_geom"]
    .apply(lambda s: s.unary_union)   # union removes overlaps
)

# 2) compute area of the union (m^2)
green_area_union_by_tract = (
    green_union_by_tract.apply(lambda geom: geom.area)
    .rename("green_area_m2_union")
    .reset_index()
)

# 3) merge back to kc (keep all 494 tracts)
kc = kc.merge(green_area_union_by_tract, on="GEOID", how="left")
kc["green_area_m2_union"] = kc["green_area_m2_union"].fillna(0.0)

# 4) recompute green_pct using union-area
kc["green_pct_union"] = np.where(
    kc["tract_area_m2"] > 0,
    kc["green_area_m2_union"] / kc["tract_area_m2"],
    np.nan
)

print("green_pct_union min/max:", kc["green_pct_union"].min(), kc["green_pct_union"].max())
print("tracts with green_pct_union > 1:", (kc["green_pct_union"] > 1).sum())

kc.loc[kc["green_pct_union"] > 1, ["GEOID", "green_area_m2_union", "tract_area_m2", "green_pct_union"]].head(10)


C:\Users\yazed\AppData\Local\Temp\ipykernel_35228\2745945131.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .apply(lambda s: s.unary_union)   # union removes overlaps


green_pct_union min/max: 0.0 0.8737408837234383
tracts with green_pct_union > 1: 0


,GEOID,green_area_m2_union,tract_area_m2,green_pct_union


In [182]:
# Make union-based greenspace the official features
kc["green_area_m2"] = kc["green_area_m2_union"]
kc["green_pct"] = kc["green_pct_union"]

# Optional cleanup: drop the temporary union columns
kc = kc.drop(columns=["green_area_m2_union", "green_pct_union"], errors="ignore")

# Sanity checks (must be true)
print("rows:", len(kc), "| unique GEOID:", kc["GEOID"].nunique())
print("green_pct min/max:", kc["green_pct"].min(), kc["green_pct"].max())
print("tracts with green_pct > 1:", (kc["green_pct"] > 1).sum())
print("tracts with green_pct < 0:", (kc["green_pct"] < 0).sum())

rows: 494 | unique GEOID: 494
green_pct min/max: 0.0 0.8737408837234383
tracts with green_pct > 1: 0
tracts with green_pct < 0: 0


In [183]:
kc.head()

,GEOID,asthma_prev,pop_total,pop_18plus,asthma_ci_low,asthma_ci_high,geometry,roads_length_m_drive,tract_area_m2,green_pct,green_area_m2
0,53033030003,11.1,6592,5347,10.3,12.0,"POLYGON ((550504.065 5246729.293, 550986.386 5...",42584.740622,5.563658e+06,0.236221,1.314252e+06
1,53033005307,13.9,2921,2824,12.7,15.3,"POLYGON ((551742.634 5279083.437, 551742.949 5...",2090.239238,1.299323e+05,0.022185,2.882592e+03
2,53033030309,10.6,5906,4666,9.8,11.5,"POLYGON ((549077.056 5238559.971, 549167.882 5...",22296.721590,2.219825e+06,0.243334,5.401581e+05
3,53033030310,9.6,6466,4998,8.8,10.4,"POLYGON ((548308.414 5237189.39, 548308.772 52...",32094.396080,3.203117e+06,0.133068,4.262330e+05
4,53033030311,10.4,5093,3810,9.6,11.2,"POLYGON ((547093.219 5239575.488, 547093.752 5...",23152.827072,1.903031e+06,0.076483,1.455496e+05


In [184]:
kc.to_file(str(data_dir) + "/processed/" + "king_county_tracts_asthma_roads_greenspace_v01_epsg26910.geojson", driver='GeoJSON')

In [5]:
# read kc dataset

kc = gpd.read_file(f"{data_dir}/processed/king_county_tracts_asthma_roads_greenspace_v01_epsg26910.geojson")
kc.head()

,GEOID,asthma_prev,pop_total,pop_18plus,asthma_ci_low,asthma_ci_high,roads_length_m_drive,tract_area_m2,green_pct,green_area_m2,geometry
0,53033030003,11.1,6592,5347,10.3,12.0,42584.740622,5.563658e+06,0.236221,1.314252e+06,"POLYGON ((550504.065 5246729.293, 550986.386 5..."
1,53033005307,13.9,2921,2824,12.7,15.3,2090.239238,1.299323e+05,0.022185,2.882592e+03,"POLYGON ((551742.634 5279083.437, 551742.949 5..."
2,53033030309,10.6,5906,4666,9.8,11.5,22296.721590,2.219825e+06,0.243334,5.401581e+05,"POLYGON ((549077.056 5238559.971, 549167.882 5..."
3,53033030310,9.6,6466,4998,8.8,10.4,32094.396080,3.203117e+06,0.133068,4.262330e+05,"POLYGON ((548308.414 5237189.39, 548308.772 52..."
4,53033030311,10.4,5093,3810,9.6,11.2,23152.827072,1.903031e+06,0.076483,1.455496e+05,"POLYGON ((547093.219 5239575.488, 547093.752 5..."


In [6]:
# adding tract_area_km2 & road_km_per_km2 columns 
kc["tract_area_km2"] = kc["tract_area_m2"] / 1_000_000
kc["road_km_per_km2"] = (kc["roads_length_m_drive"] / 1_000) / \
                        kc["tract_area_km2"]

kc.head()

,GEOID,asthma_prev,pop_total,pop_18plus,asthma_ci_low,asthma_ci_high,roads_length_m_drive,tract_area_m2,green_pct,green_area_m2,geometry,tract_area_km2,road_km_per_km2
0,53033030003,11.1,6592,5347,10.3,12.0,42584.740622,5.563658e+06,0.236221,1.314252e+06,"POLYGON ((550504.065 5246729.293, 550986.386 5...",5.563658,7.654090
1,53033005307,13.9,2921,2824,12.7,15.3,2090.239238,1.299323e+05,0.022185,2.882592e+03,"POLYGON ((551742.634 5279083.437, 551742.949 5...",0.129932,16.087138
2,53033030309,10.6,5906,4666,9.8,11.5,22296.721590,2.219825e+06,0.243334,5.401581e+05,"POLYGON ((549077.056 5238559.971, 549167.882 5...",2.219825,10.044361
3,53033030310,9.6,6466,4998,8.8,10.4,32094.396080,3.203117e+06,0.133068,4.262330e+05,"POLYGON ((548308.414 5237189.39, 548308.772 52...",3.203117,10.019740
4,53033030311,10.4,5093,3810,9.6,11.2,23152.827072,1.903031e+06,0.076483,1.455496e+05,"POLYGON ((547093.219 5239575.488, 547093.752 5...",1.903031,12.166291


In [7]:
# Verify the results make sense
print(kc[["road_km_per_km2", "tract_area_km2"]].describe())

# Check for any impossible values
print(f"Any negative road density? {(kc['road_km_per_km2'] < 0).any()}")
print(f"Any zero tract area? {(kc['tract_area_km2'] == 0).any()}")

       road_km_per_km2  tract_area_km2
count       494.000000      494.000000
mean         11.120861       11.604354
std           5.666995       82.731847
min           0.332506        0.072977
25%           7.385329        1.286218
50%          10.313160        2.490315
75%          14.065112        4.325316
max          31.871314     1417.368479
Any negative road density? False
Any zero tract area? False


In [11]:
# Load TRI data
tri = pd.read_csv(f"{data_dir}/raw/tri/2024_us.csv")

# Filter to King County, Washington
tri_king = tri[(tri['8. ST'] == 'WA') & (tri['7. COUNTY'] == 'KING')].copy()

# Select essential columns and rename for clarity
tri_clean = tri_king[[
    '4. FACILITY NAME', 
    '12. LATITUDE', 
    '13. LONGITUDE', 
    '107. TOTAL RELEASES'
]].copy().rename(columns={
    '4. FACILITY NAME': 'facility_name',
    '12. LATITUDE': 'latitude',
    '13. LONGITUDE': 'longitude',
    '107. TOTAL RELEASES': 'total_releases'
})

# Remove any rows with missing coordinates
tri_clean = tri_clean.dropna(subset=['latitude', 'longitude'])

# Convert to GeoDataFrame
tri_gdf = gpd.GeoDataFrame(
    tri_clean,
    geometry=gpd.points_from_xy(tri_clean['longitude'], tri_clean['latitude']),
    crs='EPSG:4326'  # WGS84 coordinate system
)

# Reproject to match your tract data's CRS
tri_gdf = tri_gdf.to_crs(kc.crs)

print(f"Found {len(tri_gdf)} TRI facilities in King County")
print(tri_gdf.head())

Found 174 TRI facilities in King County
                                  facility_name   latitude   longitude  \
1735                   EXOTIC METALS FORMING CO  47.399767 -122.264766   
2832                       PUGET SOUND COATINGS  47.520597 -122.323010   
3437        SHELL SEATTLE DISTRIBUTION TERMINAL  47.585044 -122.352316   
3853                       PUGET SOUND COATINGS  47.520597 -122.323010   
4074  BOEING COMMERCIAL AIRPLANE GROUP - RENTON  47.498878 -122.205184   

      total_releases                        geometry  
1735         264.180  POINT (555478.174 5249852.348)  
2832        5800.000  POINT (550966.392 5263240.881)  
3437           0.000  POINT (548700.386 5270384.486)  
3853       14814.000  POINT (550966.392 5263240.881)  
4074       12944.872   POINT (559861.43 5260911.162)  


C:\Users\yazed\AppData\Local\Temp\ipykernel_10592\1558860058.py:2: DtypeWarning: Columns (0: 16. PARENT CO DB NUM) have mixed types. Specify dtype option on import or set low_memory=False.
  tri = pd.read_csv(f"{data_dir}/raw/tri/2024_us.csv")


In [23]:
kc['dist_tri_km'] = kc.geometry.apply(
    lambda r : tri_gdf.geometry.distance(r).min() / 1000
)

In [72]:
tri_geom = tri_gdf.geometry

def count_tri_in_buffer(geom):
    """
    receive a tracte buffer geometry and count how many tri in it
    """
    return sum(geom.contains(tri_geom))


def tri_releases_sum(geom):
    """receive a tracte buffer geometry and sum the total tri releases within it"""
    mask = geom.contains(tri_geom)
    tri_within = tri_gdf[mask]
    return sum(tri_within.total_releases.to_list())

In [ ]:
kc["buffer_2km"] = kc.geometry.buffer(2000)
kc["tri_count_within_buffer"] = kc.buffer_2km.apply(count_tri_in_buffer)

In [67]:
kc[["GEOID", "buffer_2km", "tri_count_within_buffer"]].sample(7)

,GEOID,buffer_2km,tri_count_within_buffer
188,53033022005,"POLYGON ((557421.153 5284575.313, 557420.994 5...",0
377,53033030403,"POLYGON ((548917.683 5234875.13, 548923.473 52...",0
242,53033021802,"POLYGON ((558647.086 5291225.364, 558638.912 5...",3
394,53033006900,"POLYGON ((545122.728 5275801.34, 545104.21 527...",2
108,53033029508,"POLYGON ((559376.984 5248460.329, 559384.406 5...",0
249,53033032320,"POLYGON ((561509.129 5288955.75, 561511.369 52...",3
375,53033025001,"POLYGON ((560079.249 5266166.263, 560079.294 5...",0


In [69]:
print(kc['tri_count_within_buffer'].describe())
print(f"Tracts with 0 facilities in 2km: {(kc['tri_count_within_buffer'] == 0).sum()}")
print(f"Max facilities in 2km: {kc['tri_count_within_buffer'].max()}")

count    494.000000
mean       3.087045
std        7.696744
min        0.000000
25%        0.000000
50%        0.000000
75%        2.000000
max       67.000000
Name: tri_count_within_buffer, dtype: float64
Tracts with 0 facilities in 2km: 283
Max facilities in 2km: 67


In [71]:
tri_gdf.head()

,facility_name,latitude,longitude,total_releases,geometry
1735,EXOTIC METALS FORMING CO,47.399767,-122.264766,264.180,POINT (555478.174 5249852.348)
2832,PUGET SOUND COATINGS,47.520597,-122.323010,5800.000,POINT (550966.392 5263240.881)
3437,SHELL SEATTLE DISTRIBUTION TERMINAL,47.585044,-122.352316,0.000,POINT (548700.386 5270384.486)
3853,PUGET SOUND COATINGS,47.520597,-122.323010,14814.000,POINT (550966.392 5263240.881)
4074,BOEING COMMERCIAL AIRPLANE GROUP - RENTON,47.498878,-122.205184,12944.872,POINT (559861.43 5260911.162)


In [79]:
kc["tri_releases_2km"] = kc.buffer_2km.apply(tri_releases_sum)

In [82]:
kc[["GEOID","dist_tri_km", "buffer_2km", "tri_count_within_buffer", "tri_releases_2km"]].sample(8)

,GEOID,dist_tri_km,buffer_2km,tri_count_within_buffer,tri_releases_2km
188,53033022005,3.926002,"POLYGON ((557421.153 5284575.313, 557420.994 5...",0,0.000
61,53033029702,0.951134,"POLYGON ((554171.151 5245964.1, 554198.387 524...",4,24915.480
151,53033007002,2.079032,"POLYGON ((546234.217 5275115.193, 546235.387 5...",0,0.000
211,53033011601,3.579420,"POLYGON ((543747.392 5264880.99, 543748.517 52...",0,0.000
94,53033025303,0.000000,"POLYGON ((557700.119 5260281.623, 557721.517 5...",9,42738.169
432,53033009702,1.959795,"POLYGON ((542817.387 5269665.815, 542817.408 5...",9,281368.848
251,53033032402,8.269442,"POLYGON ((573131.422 5284499.847, 573179.109 5...",0,0.000
67,53033032307,2.425439,"POLYGON ((564333.957 5287484.431, 564333.84 52...",0,0.000


In [84]:
# Summary stats
print("=== TRI FEATURES SUMMARY ===")
print(kc[['dist_tri_km', 'tri_count_within_buffer', 'tri_releases_2km']].describe())

# Logical checks
print(f"Any negative distances? {(kc['dist_tri_km'] < 0).any()}")
print(f"Min distance: {kc['dist_tri_km'].min():.2f} km")
print(f"Max distance: {kc['dist_tri_km'].max():.2f} km")
print(f"Tracts with facilities IN them (dist=0): {(kc['dist_tri_km'] == 0).sum()}")

=== TRI FEATURES SUMMARY ===
       dist_tri_km  tri_count_within_buffer  tri_releases_2km
count   494.000000               494.000000        494.000000
mean      2.902475                 3.087045      15734.423089
std       2.674676                 7.696744      49286.906646
min       0.000000                 0.000000          0.000000
25%       1.088616                 0.000000          0.000000
50%       2.418586                 0.000000          0.000000
75%       3.694444                 2.000000       6437.466000
max      22.060179                67.000000     331272.362000
Any negative distances? False
Min distance: 0.00 km
Max distance: 22.06 km
Tracts with facilities IN them (dist=0): 29


In [ ]:
# third-party library that helps download and work with U.S. Census Bureau data
import censusdata

# King County, WA = state 53, county 033
# Get data for all tracts in King County

# Define which variables you want (these are ACS 5-year codes)
variables = {
    'B17010_002E': 'low_income_count',     # Below poverty
    'B17010_001E': 'total_pop_poverty',    # Total pop for poverty calc
    'B03002_001E': 'total_pop_race',       # Total population
    'B03002_003E': 'white_alone',          # White alone, not Hispanic
    'B15003_001E': 'total_25plus',         # Total 25+ for education
    'B15003_002E': 'no_school',            # No schooling
    'B15003_003E': 'nursery_4th',          # Nursery to 4th
    'B15003_004E': 'grade_5_6',            # 5th-6th grade
    'B15003_005E': 'grade_7_8',            # 7th-8th grade
    'B15003_006E': 'grade_9',              # 9th grade
    'B15003_007E': 'grade_10',             # 10th grade
    'B15003_008E': 'grade_11',             # 11th grade
    'B15003_009E': 'grade_12_no_diploma',  # 12th no diploma
}

# Download data for King County tracts (2022 ACS 5-year)
acs_data = censusdata.download(
    'acs5', 2022,
    censusdata.censusgeo([('state', '53'), ('county', '033'), ('tract', '*')]),
    list(variables.keys())
)

# Rename columns
acs_data.columns = variables.values()

# Add GEOID
acs_data['GEOID'] = acs_data.index.map(lambda x: x.geo[0][1] + x.geo[1][1] + x.geo[2][1])

print(acs_data.head())

                                                    low_income_count  \
Census Tract 1.01; King County; Washington: Sum...                69   
Census Tract 1.02; King County; Washington: Sum...               148   
Census Tract 2.01; King County; Washington: Sum...               179   
Census Tract 2.02; King County; Washington: Sum...                 0   
Census Tract 3; King County; Washington: Summar...                10   

                                                    total_pop_poverty  \
Census Tract 1.01; King County; Washington: Sum...                689   
Census Tract 1.02; King County; Washington: Sum...                975   
Census Tract 2.01; King County; Washington: Sum...                995   
Census Tract 2.02; King County; Washington: Sum...               1028   
Census Tract 3; King County; Washington: Summar...                712   

                                                    total_pop_race  \
Census Tract 1.01; King County; Washington: Sum...        

In [90]:
# Reset and extract GEOID correctly
acs_data_reset = acs_data.reset_index()

# The GEOID is built from state + county + tract codes
# For King County, WA: state=53, county=033
acs_data_reset['GEOID'] = (
    acs_data_reset['index'].astype(str)
    .str.extract(r'tract:(\d+)')[0]  # Extract tract number
    .str.zfill(6)  # Pad with zeros to 6 digits
)

# Add state + county prefix
acs_data_reset['GEOID'] = '53033' + acs_data_reset['GEOID']

print(acs_data_reset['GEOID'].head())

0    53033000101
1    53033000102
2    53033000201
3    53033000202
4    53033000300
Name: GEOID, dtype: str


In [ ]:
# Low income percentage
acs_data_reset['low_income_pct'] = (acs_data_reset['low_income_count'] / acs_data_reset['total_pop_poverty']) * 100

# Minority percentage (everyone who is NOT white alone)
acs_data_reset['minority_pct'] = ((acs_data_reset['total_pop_race'] - acs_data_reset['white_alone']) / acs_data_reset['total_pop_race']) * 100

# Education - less than high school
less_hs_cols = ['no_school', 'nursery_4th', 'grade_5_6', 'grade_7_8', 
                'grade_9', 'grade_10', 'grade_11', 'grade_12_no_diploma']
acs_data_reset['education_low_pct'] = (acs_data_reset[less_hs_cols].sum(axis=1) / acs_data_reset['total_25plus']) * 100

# Keep only what i need
controls = acs_data_reset[['GEOID', 'low_income_pct', 'minority_pct', 'education_low_pct']]

In [94]:
controls.isna().sum()

GEOID                0
low_income_pct       2
minority_pct         1
education_low_pct    1
dtype: int64

In [97]:
controls.head()

,GEOID,low_income_pct,minority_pct,education_low_pct
0,53033000101,10.014514,54.192410,1.038462
1,53033000102,15.179487,24.828845,2.814977
2,53033000201,17.989950,41.441057,2.352623
3,53033000202,0.000000,48.519774,0.883338
4,53033000300,1.404494,36.369771,2.383420


In [98]:
kc = kc.merge(controls, on='GEOID', how='left')

print("\n=== MERGE COMPLETE ===")
print(f"Total tracts: {len(kc)}")
print(f"Missing low_income_pct: {kc['low_income_pct'].isna().sum()}")
print(f"Missing minority_pct: {kc['minority_pct'].isna().sum()}")
print(f"Missing education_low_pct: {kc['education_low_pct'].isna().sum()}")


=== MERGE COMPLETE ===
Total tracts: 494
Missing low_income_pct: 1
Missing minority_pct: 0
Missing education_low_pct: 0


In [99]:
# Summary of all your variables
print("\n=== COMPLETE FEATURE SET ===")
print(kc[['asthma_prev', 'dist_tri_km', 'tri_count_within_buffer', 
          'road_km_per_km2', 'green_pct',
          'low_income_pct', 'minority_pct', 'education_low_pct']].describe())

# Sample rows to verify
print("\nSample of complete data:")
print(kc[['GEOID', 'asthma_prev', 'dist_tri_km', 'tri_count_within_buffer',
          'low_income_pct', 'minority_pct']].head())


=== COMPLETE FEATURE SET ===
       asthma_prev  dist_tri_km  tri_count_within_buffer  road_km_per_km2  \
count   494.000000   494.000000               494.000000       494.000000   
mean      9.840081     2.902475                 3.087045        11.120861   
std       0.951700     2.674676                 7.696744         5.666995   
min       7.300000     0.000000                 0.000000         0.332506   
25%       9.300000     1.088616                 0.000000         7.385329   
50%       9.900000     2.418586                 0.000000        10.313160   
75%      10.500000     3.694444                 2.000000        14.065112   
max      13.900000    22.060179                67.000000        31.871314   

        green_pct  low_income_pct  minority_pct  education_low_pct  
count  494.000000      493.000000    494.000000         494.000000  
mean     0.162756        5.790554     43.692569           2.109898  
std      0.158868        8.131670     17.777440           2.712288  


In [104]:
kc.head(2)

,GEOID,asthma_prev,pop_total,pop_18plus,asthma_ci_low,asthma_ci_high,roads_length_m_drive,tract_area_m2,green_pct,green_area_m2,geometry,tract_area_km2,road_km_per_km2,dist_tri_km,buffer_2km,tri_count_within_buffer,tri_releases_2km,low_income_pct,minority_pct,education_low_pct
0,53033030003,11.1,6592,5347,10.3,12.0,42584.740622,5.563658e+06,0.236221,1.314252e+06,"POLYGON ((550504.065 5246729.293, 550986.386 5...",5.563658,7.654090,0.144760,"POLYGON ((548552.328 5246292.576, 548518.784 5...",1,8917.0,6.76819,50.067861,5.067706
1,53033005307,13.9,2921,2824,12.7,15.3,2090.239238,1.299323e+05,0.022185,2.882592e+03,"POLYGON ((551742.634 5279083.437, 551742.949 5...",0.129932,16.087138,3.331304,"POLYGON ((549742.636 5279086.498, 549742.952 5...",0,0.0,NaN,45.805593,0.000000


In [105]:
# Drop unnecessary columns and Step 2: Rename to clean names

kc_clean = kc[[
    'GEOID',
    'geometry',
    'asthma_prev', 
    'asthma_ci_low', 
    'asthma_ci_high',
    'pop_total', 
    'pop_18plus',
    'dist_tri_km', 
    'tri_count_within_buffer', 
    'tri_releases_2km',
    'road_km_per_km2',
    'green_pct',
    'tract_area_km2',
    'low_income_pct', 
    'minority_pct', 
    'education_low_pct'
]].copy()

# Rename to better names
kc_clean = kc_clean.rename(columns={
    'GEOID': 'geoid',
    'asthma_prev': 'asthma_prevalence',
    'asthma_ci_low': 'asthma_ci_lower',
    'asthma_ci_high': 'asthma_ci_upper',
    'pop_total': 'population_total',
    'pop_18plus': 'population_adult',
    'dist_tri_km': 'distance_tri_km',
    'tri_count_within_buffer': 'tri_count_2km',
    'tri_releases_2km': 'tri_total_releases_2km',
    'road_km_per_km2': 'road_density_km_per_km2',
    'green_pct': 'green_space_percent',
    'tract_area_km2': 'area_km2',
    'low_income_pct': 'percent_low_income',
    'minority_pct': 'percent_minority',
    'education_low_pct': 'percent_less_than_hs'
})

# Save
output_file = f"{data_dir}/processed/king_tracts_analysis.geojson"
kc_clean.to_file(output_file, driver='GeoJson')

print("Clean analysis table saved!")
print(f"\nLocation: {output_file}")
print(f"Dataset: {len(kc_clean)} tracts × {len(kc_clean.columns)} columns")
print(f"\nFinal columns:")
for i, col in enumerate(kc_clean.columns, 1):
    print(f"  {i:2d}. {col}")

Clean analysis table saved!

Location: c:\dev\projects\respiratory_project\data/processed/king_tracts_analysis.geojson
Dataset: 494 tracts × 16 columns

Final columns:
   1. geoid
   2. geometry
   3. asthma_prevalence
   4. asthma_ci_lower
   5. asthma_ci_upper
   6. population_total
   7. population_adult
   8. distance_tri_km
   9. tri_count_2km
  10. tri_total_releases_2km
  11. road_density_km_per_km2
  12. green_space_percent
  13. area_km2
  14. percent_low_income
  15. percent_minority
  16. percent_less_than_hs


##### END.